# GCP VM MariaDB 資料庫連線語法大全

這個 Jupyter Notebook 提供使用 Python 連線到 GCP VM 上 MariaDB 資料庫的完整範例。

## 環境準備
- 需要安裝的套件：`pip install mysql-connector-python pandas sqlalchemy`
- 支援操作：連線、查詢 (SELECT)、插入 (INSERT)、更新 (UPDATE)、刪除 (DELETE)、交易處理等。
- 注意事項：
  - 確保 GCP VM 的防火牆允許 3306 連接埠。
  - 替換以下連線資訊為您自己的。
  - DATABASE 和 TABLE 可以根據需要修改。

In [5]:
# ==================== 1. 導入必要套件 ====================
# mysql-connector-python: 用於直接連線 MariaDB
# pandas: 用於資料讀取與處理
# sqlalchemy: 用於更高階的資料庫操作（如 pandas.to_sql）
import mysql.connector
from mysql.connector import Error
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime

In [6]:
# ==================== 2. 設定連線參數 ====================
# 使用外部設定檔 db_config.py
try:
    from db_config import DB_HOST, DB_USER, DB_PASSWORD, DB_NAME
except ImportError:
    print("錯誤：找不到 db_config.py")
    # 如果找不到，這裡可以留空或拋出錯誤
    DB_HOST, DB_USER, DB_PASSWORD, DB_NAME = '', '', '', ''
HOST = DB_HOST
USER = DB_USER
PASSWORD = DB_PASSWORD
DATABASE = DB_NAME
# TABLE 保留為原本的設定或不變

# 建立 SQLAlchemy 引擎（用於 pandas 操作）
engine = create_engine(f'mysql+mysqlconnector://{USER}:{PASSWORD}@{HOST}:3306/{DATABASE}')

# 測試連線是否成功
try:
    with engine.connect() as conn:
        print(f"成功連線到資料庫: {DATABASE} on {HOST}")
except Error as e:
    print(f"連線失敗: {e}")

成功連線到資料庫: rawdata on 35.185.175.249


## 範例 1: 基本連線與斷開連線

使用 `mysql.connector` 建立連線，並確保斷開連線。

In [ ]:
# ==================== 範例 1: 基本連線與斷開 ====================
try:
    # 建立連線
    conn = mysql.connector.connect(
        host=HOST,
        user=USER,
        password=PASSWORD,
        database=DATABASE
    )
    
    if conn.is_connected():
        print("連線成功！")
        cursor = conn.cursor()  # 建立游標，用於執行 SQL
        cursor.execute("SELECT VERSION()")  # 查詢 MariaDB 版本
        version = cursor.fetchone()
        print(f"MariaDB 版本: {version[0]}")
    
except Error as e:
    print(f"連線錯誤: {e}")
finally:
    if 'conn' in locals() and conn.is_connected():
        cursor.close()  # 關閉游標
        conn.close()    # 關閉連線
        print("連線已關閉")

## 範例 2: 查詢資料 (SELECT)

- 使用直接 SQL 查詢。
- 使用 pandas 讀取查詢結果。

In [4]:
TABLE = 'job_details'

# ==================== 範例 2.1: 使用 mysql-connector 執行 SELECT ====================
try:
    conn = mysql.connector.connect(host=HOST, user=USER, password=PASSWORD, database=DATABASE)
    cursor = conn.cursor()
    
    # SQL 查詢語法（可修改條件）
    query = f"SELECT * FROM {TABLE} LIMIT 5"  # 查詢前 5 筆資料
    cursor.execute(query)
    
    # 取得結果
    results = cursor.fetchall()
    for row in results:
        print(row)  # 印出每一行資料
    
except Error as e:
    print(f"查詢錯誤: {e}")
finally:
    if conn.is_connected():
        cursor.close()
        conn.close()

(1, 'https://www.cake.me/companies/commeet/jobs/data-assistant-engineerdata-science-intern', '數據工程師實習生 Data Engineer Intern', 'COMMEET＿擁樂數據服務股份有限公司', '軟體', '數據工程師', '200 ~ 250 TWD / 小時', '台北市, 台灣', '經驗不拘', '實習生', '初階', '1', '不需負擔管理責任', '不支援遠端', '松山區長安東路二段225號A棟1樓之1, 松山區, 台北', 'python, postgres, redis, git', '職缺 1 天前更新', 'COMMEET (https://commeet.co) 擴大徵才，歡迎有熱情之新夥伴加入我們的行列\n[工作內容]\n1 整理、標記數據，以確保數據的準確性與可靠性。\n2 訓練OCR模型，並進行模型效能評估。\n3 建立/維護/優化資料處理流程與架構(data pipeline)。\n4 從業人員恪守客戶個資保護敏感度及尊重客戶資料隱密性。\n5 其它主管交辦事項。\n[職缺條件]\n• 熟悉 ML/DL\n• 熟悉 python 語言與相關資料庫 (csv、postgres、redis、excel)\n• 熟悉 git 版本控制\n• 入職3個月內需通過公司內部程式能力測驗(實習生需刷題75題以上)\n[排班說明]\n• 非寒暑假短期性質必須配合「實習達六個月」。\n• 比照公務機關行事曆出勤，一週達24小時，須至辦公室出勤。\n• 每月雙方協商排定班表，若無法約定班表出勤，須提前溝通調整班表。\n• 出勤工作日排班最小單位為一個時段 (選上半天時段/下半天時段/全日時段)。\n- 全日班表定工作時間 09:30 ~18:30，可彈性於 09:00 ~ 10:00 間簽到上班，午休時段可選擇12:00-13:00 或 12:30-13:30，每日正常工作時間不超過八小時。\n- 09:30 ~ 13:30 上半天工作時段 (上班有彈性工時)，工作四小時(含)則不會有休息。\n- 14:30 ~ 18:30 下半天工作時段 (上班無彈性工時)，工作四小時(含)則不會有休息。\n初試：線上面試，請您提前準備自我介紹簡報，有效率的讓

In [3]:
# ==================== 範例 2.2: 使用 pandas 讀取 SELECT 結果 ====================
# SQL 查詢語法（可修改條件）
query = f"SELECT job_title, salary, update_date FROM {TABLE} WHERE salary LIKE '%面議%' LIMIT 10"

# 使用 pandas 直接讀取
df = pd.read_sql(query, engine)
print("查詢結果（DataFrame）：")
display(df)  # 在 Notebook 中顯示表格

查詢結果（DataFrame）：


,job_title,salary,update_date
0,技術工程類-AI自動化工程師,待遇面議,10月22日
1,JC2002-軟體開發工程師(資料庫)-新竹區,待遇面議,10月22日
2,AI研發工程師,待遇面議,10月22日
3,數據分析工程師,待遇面議,10月22日
4,【總部】程式設計師,待遇面議,10月22日
5,AI Application Engineer(竹南/台北),待遇面議,10月22日
6,AI應用工程師 AI Application Engineer,待遇面議,10月22日
7,AI資料與分析工程師 AI Data & Analytics Engineer,待遇面議,10月22日
8,資料分析師 / Data Analyst - Tableau,待遇面議,10月22日
9,AI 演算法工程師,待遇面議,10月22日


## 範例 3: 插入資料 (INSERT)

- 單筆插入。
- 多筆插入。

In [ ]:
# ==================== 範例 3.1: 單筆插入 ====================
try:
    conn = mysql.connector.connect(host=HOST, user=USER, password=PASSWORD, database=DATABASE)
    cursor = conn.cursor()
    
    # SQL 插入語法（假設 TABLE 有 id, job_title, salary 欄位；請根據實際 TABLE 調整）
    insert_query = f"INSERT INTO {TABLE} (job_title, salary, update_date) VALUES (%s, %s, %s)"
    data = ("測試職缺", "月薪 50,000", datetime.now().strftime('%Y-%m-%d'))  # 資料值
    
    cursor.execute(insert_query, data)
    conn.commit()  # 提交變更
    print(f"成功插入 1 筆資料，ID: {cursor.lastrowid}")
    
except Error as e:
    print(f"插入錯誤: {e}")
finally:
    if conn.is_connected():
        cursor.close()
        conn.close()

In [ ]:
# ==================== 範例 3.2: 多筆插入 ====================
try:
    conn = mysql.connector.connect(host=HOST, user=USER, password=PASSWORD, database=DATABASE)
    cursor = conn.cursor()
    
    insert_query = f"INSERT INTO {TABLE} (job_title, salary, update_date) VALUES (%s, %s, %s)"
    data_list = [
        ("測試職缺1", "月薪 60,000", datetime.now().strftime('%Y-%m-%d')),
        ("測試職缺2", "月薪 70,000", datetime.now().strftime('%Y-%m-%d'))
    ]
    
    cursor.executemany(insert_query, data_list)
    conn.commit()
    print(f"成功插入 {cursor.rowcount} 筆資料")
    
except Error as e:
    print(f"插入錯誤: {e}")
finally:
    if conn.is_connected():
        cursor.close()
        conn.close()

## 範例 4: 更新資料 (UPDATE)

更新指定條件的資料。

In [ ]:
# ==================== 範例 4: 更新資料 ====================
try:
    conn = mysql.connector.connect(host=HOST, user=USER, password=PASSWORD, database=DATABASE)
    cursor = conn.cursor()
    
    # SQL 更新語法（假設有 id 欄位；請根據實際調整）
    update_query = f"UPDATE {TABLE} SET salary = %s WHERE job_title = %s"
    data = ("月薪 80,000", "測試職缺")  # 更新值與條件
    
    cursor.execute(update_query, data)
    conn.commit()
    print(f"成功更新 {cursor.rowcount} 筆資料")
    
except Error as e:
    print(f"更新錯誤: {e}")
finally:
    if conn.is_connected():
        cursor.close()
        conn.close()

## 範例 5: 刪除資料 (DELETE)

刪除指定條件的資料。

In [ ]:
# ==================== 範例 5: 刪除資料 ====================
try:
    conn = mysql.connector.connect(host=HOST, user=USER, password=PASSWORD, database=DATABASE)
    cursor = conn.cursor()
    
    # SQL 刪除語法（請小心使用，避免誤刪）
    delete_query = f"DELETE FROM {TABLE} WHERE job_title = %s"
    data = ("測試職缺",)  # 條件
    
    cursor.execute(delete_query, data)
    conn.commit()
    print(f"成功刪除 {cursor.rowcount} 筆資料")
    
except Error as e:
    print(f"刪除錯誤: {e}")
finally:
    if conn.is_connected():
        cursor.close()
        conn.close()

## 範例 6: 交易處理 (Transaction) - Commit & Rollback

使用交易確保資料一致性。

In [ ]:
# ==================== 範例 6: 交易處理 ====================
try:
    conn = mysql.connector.connect(host=HOST, user=USER, password=PASSWORD, database=DATABASE)
    conn.autocommit = False  # 關閉自動提交，啟用手動交易
    cursor = conn.cursor()
    
    # 執行多個操作
    insert_query = f"INSERT INTO {TABLE} (job_title, salary, update_date) VALUES (%s, %s, %s)"
    cursor.execute(insert_query, ("交易測試", "月薪 90,000", datetime.now().strftime('%Y-%m-%d')))
    
    # 模擬錯誤（取消註解來測試 rollback）
    # raise Error("模擬錯誤")
    
    conn.commit()  # 成功則提交
    print("交易成功提交")
    
except Error as e:
    conn.rollback()  # 錯誤則回滾
    print(f"交易失敗，回滾: {e}")
finally:
    if conn.is_connected():
        cursor.close()
        conn.close()

## 範例 7: 使用 pandas 寫入資料 (to_sql)

將 DataFrame 直接寫入資料庫。

In [ ]:
# ==================== 範例 7: 使用 pandas 寫入資料 ====================
# 建立一個測試 DataFrame
data = {
    'job_title': ['Pandas測試1', 'Pandas測試2'],
    'salary': ['月薪 100,000', '月薪 110,000'],
    'update_date': [datetime.now().strftime('%Y-%m-%d'), datetime.now().strftime('%Y-%m-%d')]
}
df_write = pd.DataFrame(data)

# 寫入資料庫（if_exists='append' 表示附加，不覆蓋）
df_write.to_sql(TABLE, engine, if_exists='append', index=False)
print(f"成功寫入 {len(df_write)} 筆資料到 {TABLE}")

## 範例 8: 進階查詢 - 帶參數的查詢 (防止 SQL Injection)

使用參數化查詢以提升安全性。

In [ ]:
# ==================== 範例 8: 帶參數的查詢 ====================
try:
    conn = mysql.connector.connect(host=HOST, user=USER, password=PASSWORD, database=DATABASE)
    cursor = conn.cursor()
    
    # 參數化 SQL（防止 SQL Injection）
    query = f"SELECT * FROM {TABLE} WHERE job_title LIKE %s LIMIT 5"
    param = ("%測試%",)  # 查詢包含 '測試' 的職缺
    
    cursor.execute(query, param)
    results = cursor.fetchall()
    for row in results:
        print(row)
    
except Error as e:
    print(f"查詢錯誤: {e}")
finally:
    if conn.is_connected():
        cursor.close()
        conn.close()

## 範例 9: 建立新資料庫或表格 (CREATE)

示範建立新資料庫或表格。

In [ ]:
# ==================== 範例 9.1: 建立新資料庫 ====================
try:
    conn = mysql.connector.connect(host=HOST, user=USER, password=PASSWORD)
    cursor = conn.cursor()
    
    new_db = "test_database"  # 新資料庫名稱
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS {new_db}")
    print(f"成功建立資料庫: {new_db}")
    
except Error as e:
    print(f"建立錯誤: {e}")
finally:
    if conn.is_connected():
        cursor.close()
        conn.close()

In [ ]:
# ==================== 範例 9.2: 建立新表格 ====================
try:
    conn = mysql.connector.connect(host=HOST, user=USER, password=PASSWORD, database=DATABASE)
    cursor = conn.cursor()
    
    new_table = "test_table"  # 新表格名稱
    create_query = f"""
        CREATE TABLE IF NOT EXISTS {new_table} (
            id INT AUTO_INCREMENT PRIMARY KEY,
            name VARCHAR(255),
            created_at DATETIME
        )
    """
    cursor.execute(create_query)
    print(f"成功建立表格: {new_table}")
    
except Error as e:
    print(f"建立錯誤: {e}")
finally:
    if conn.is_connected():
        cursor.close()
        conn.close()

## 範例 10: 資料庫結構檢查 (Database Inspection)

包含以下檢查項目：
- 列出所有資料庫 (SHOW DATABASES)
- 列出目前資料庫的所有表格 (SHOW TABLES)
- 檢查每個表格的資料筆數 (Row Count)
- 檢查每個表格的欄位資訊 (Schema/Columns)

In [7]:
# ==================== 範例 10: 資料庫結構檢查 ====================
try:
    # 1. 連線到 Server (不指定 Database 以便查看所有 DB)
    # 注意：需使用之前定義好的 HOST, USER, PASSWORD 變數
    conn = mysql.connector.connect(host=HOST, user=USER, password=PASSWORD)
    cursor = conn.cursor()

    print("\n--- 1. 列出所有資料庫 (SHOW DATABASES) ---")
    cursor.execute("SHOW DATABASES")
    dbs = cursor.fetchall()
    df_dbs = pd.DataFrame(dbs, columns=['Database Name'])
    display(df_dbs)

    # 2. 切換到指定資料庫並列出表格
    conn.database = DATABASE
    print(f"\n--- 2. 列出資料庫 '{DATABASE}' 中的所有表格 (SHOW TABLES) ---")
    cursor.execute("SHOW TABLES")
    tables = cursor.fetchall()
    # tables 是一個 list of tuples e.g., [('table1',), ('table2',)]
    table_names = [t[0] for t in tables]
    df_tables = pd.DataFrame(table_names, columns=['Table Name'])
    display(df_tables)

    # 3. 檢查每個表格的詳細資訊
    print("\n--- 3. 檢查每個表格的資料筆數與欄位資訊 ---")
    for table in table_names:
        print(f"\n========================================")
        print(f"表格名稱: {table}")
        
        # 查詢筆數
        cursor.execute(f"SELECT COUNT(*) FROM {table}")
        count = cursor.fetchone()[0]
        print(f"資料筆數: {count}")
        
        # 查詢欄位資訊 (使用 pandas read_sql 讀取 DESCRIBE)
        print("欄位結構:")
        df_schema = pd.read_sql(f"DESCRIBE {table}", conn)
        display(df_schema)

except Error as e:
    print(f"檢查錯誤: {e}")
finally:
    if 'conn' in locals() and conn.is_connected():
        cursor.close()
        conn.close()
        print("\n檢查完成，連線已關閉")


--- 1. 列出所有資料庫 (SHOW DATABASES) ---


,Database Name
0,information_schema
1,rawdata



--- 2. 列出資料庫 'rawdata' 中的所有表格 (SHOW TABLES) ---


,Table Name
0,104rawdata
1,all_jobs_master
2,job_details



--- 3. 檢查每個表格的資料筆數與欄位資訊 ---

表格名稱: 104rawdata
資料筆數: 13884
欄位結構:


...

,Field,Type,Null,Key,Default,Extra
0,title,varchar(255),YES,,None,
1,company,varchar(255),YES,,None,
2,industry,varchar(255),YES,,None,
3,salary_raw,varchar(255),YES,,None,
4,avg_salary,int(11),YES,,None,
5,location,varchar(255),YES,,None,
6,content,text,YES,,None,
7,source,varchar(50),YES,,None,
8,url,text,YES,,None,



表格名稱: job_details
資料筆數: 1017
欄位結構:


C:\Users\fordi\AppData\Local\Temp\ipykernel_51784\4262808631.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_schema = pd.read_sql(f"DESCRIBE {table}", conn)


,Field,Type,Null,Key,Default,Extra
0,id,int(11),NO,PRI,None,auto_increment
1,url,text,YES,,None,
2,job_title,varchar(255),YES,,None,
3,company,varchar(255),YES,,None,
4,industry,varchar(255),YES,,None,
5,category,varchar(255),YES,,None,
6,salary,varchar(255),YES,,None,
7,location,varchar(255),YES,,None,
8,experience,varchar(255),YES,,None,
9,job_type,varchar(50),YES,,None,



檢查完成，連線已關閉


## 注意事項

- **安全性**：避免在程式碼中硬編碼密碼，建議使用環境變數或 secrets 管理。
- **錯誤處理**：每個範例都有 try-except 來捕捉錯誤。
- **效能**：對於大量資料，使用 pandas 或 chunksize 參數來分批處理。
- **自訂**：根據您的 TABLE 結構調整 SQL 語法中的欄位名稱。